In [36]:
import tensorflow as tf
import numpy as np

fac = 5
Mnist = tf.keras.datasets.mnist

class Linear_Layer:
    def __init__(self, in_dim, out_dim, alpha = 0.01, Theta = None, bias = None):
        self.alpha = alpha
        if Theta == None:
            self.Theta = np.random.randn(in_dim, out_dim)/fac
        else:
            self.Theta = Theta
        if bias == None:
            self.bias = np.random.randn(out_dim)/fac
        else:
            self.bias = bias
        #self.X.shape = (batch, channels, height, width)
    def forward_pass(self, X):
        self.X = X
        #1. Forward pass
        self.z = np.matmul(X, self.Theta) + self.bias
        return self.z
    def backprop(self, grad_previous):
        # batch size
        t= self.X.shape[0]
        self.grad = np.matmul(self.X.T, grad_previous) / t
        self.grad_bias = grad_previous.sum(axis=0) / t
        self.grad_a = np.matmul(grad_previous, self.Theta.transpose())
        return self.grad_a
    def applying_sgd(self):
            self.Theta = self.Theta - (self.alpha*self.grad)
            self.bias = self.bias - (self.alpha*self.grad_bias)

In [37]:
class ConvLayer:
    def __init__(self, filter_dim=3, num_kernels=8, stride=1, pad=0, alpha=0.01):
        self.filter_dim = filter_dim
        self.num_kernels = num_kernels
        self.stride = stride
        self.pad = pad
        self.alpha = alpha
        self.W = None
        self.b = None

    def _init_params(self, in_channels):
        scale = np.sqrt(2.0 / (in_channels * self.filter_dim * self.filter_dim))
        self.W = np.random.randn(self.num_kernels, in_channels, self.filter_dim, self.filter_dim) * scale
        self.b = np.zeros(self.num_kernels)

    def forward_pass(self, X):
        self.X = X
        N, C, H, W = X.shape

        if self.W is None:
            self._init_params(C)

        if self.pad > 0:
            Xp = np.pad(X, ((0, 0), (0, 0), (self.pad, self.pad), (self.pad, self.pad)), mode='constant')
        else:
            Xp = X

        self.Xp = Xp
        H_out = (Xp.shape[2] - self.filter_dim) // self.stride + 1
        W_out = (Xp.shape[3] - self.filter_dim) // self.stride + 1
        out = np.zeros((N, self.num_kernels, H_out, W_out))

        for n in range(N):
            for k in range(self.num_kernels):
                for i in range(H_out):
                    hs = i * self.stride
                    for j in range(W_out):
                        ws = j * self.stride
                        region = Xp[n, :, hs:hs + self.filter_dim, ws:ws + self.filter_dim]
                        out[n, k, i, j] = np.sum(region * self.W[k]) + self.b[k]

        self.z = out
        return out

    def backprop(self, grad_z):
        N, K, H_out, W_out = grad_z.shape
        _, C, H_pad, W_pad = self.Xp.shape

        dXp = np.zeros_like(self.Xp)
        dW = np.zeros_like(self.W)
        db = np.zeros_like(self.b)

        for n in range(N):
            for k in range(K):
                for i in range(H_out):
                    hs = i * self.stride
                    for j in range(W_out):
                        ws = j * self.stride
                        g = grad_z[n, k, i, j]
                        region = self.Xp[n, :, hs:hs + self.filter_dim, ws:ws + self.filter_dim]

                        dXp[n, :, hs:hs + self.filter_dim, ws:ws + self.filter_dim] += self.W[k] * g
                        dW[k] += region * g
                        db[k] += g

        if self.pad > 0:
            dX = dXp[:, :, self.pad:H_pad - self.pad, self.pad:W_pad - self.pad]
        else:
            dX = dXp

        self.grad_filter = dW / N
        self.grad_bias = db / N
        return dX

    def applying_sgd(self):
        self.W -= self.alpha * self.grad_filter
        self.b -= self.alpha * self.grad_bias

    def change_alpha(self):
        self.alpha = self.alpha / 10

In [38]:
class Pooling:
    def __init__(self, pool_dim=2, stride=2):
        self.pool_dim = pool_dim
        self.stride = stride
    
    def forward_pass(self, data):
        self.data = data
        N, C, H, W = data.shape
        H_out = (H - self.pool_dim) // self.stride + 1
        W_out = (W - self.pool_dim) // self.stride + 1
        out = np.zeros((N, C, H_out, W_out))

        for n in range(N):
            for c in range(C):
                for i in range(H_out):
                    for j in range(W_out):
                        out[n, c, i, j] = np.max(data[n, c, i*self.stride:i*self.stride+self.pool_dim, j*self.stride:j*self.stride+self.pool_dim])
        return out
        # z_x = int((p - self.pool_dim) / self.stride) + 1
        # z_y = int((t - self.pool_dim) / self.stride) + 1
        # after_pool = np.zeros((q, z_x, z_y))
        # for ii in range(0, q):
        #     liss = []
        #     for i in range(0, p, self.stride):
        #         for j in range(0, t, self.stride):
        #             if (i + self.pool_dim <= p) and (j + self.pool_dim <= t):
        #                 temp = data[ii, i:i + self.pool_dim, j:j + self.pool_dim]
        #                 temp_1 = np.max(temp)
        #                 liss.append(temp_1)
        #     liss = np.asarray(liss)
        #     liss = liss.reshape((z_x, z_y))
        #     after_pool[ii] = liss
        #     del liss
        # return after_pool
    
    def backprop(self, grad_out):
        N, C, H_out, W_out = grad_out.shape
        grad_in = np.zeros_like(self.data)

        for n in range(N):
            for c in range(C):
                for i in range(H_out):
                    for j in range(W_out):
                        pool_region = self.data[n, c, i*self.stride:i*self.stride+self.pool_dim, j*self.stride:j*self.stride+self.pool_dim]
                        max_val = np.max(pool_region)
                        grad_in[n, c, i*self.stride:i*self.stride+self.pool_dim, j*self.stride:j*self.stride+self.pool_dim] += (pool_region == max_val) * grad_out[n, c, i, j]
        return grad_in

        # cheated = np.zeros((a, 2 * b, 2 * c))
        # for k in range(0, a):
        #     pooled_transpose_re = pooled[k].reshape((b * c))
        #     count = 0
        #     for i in range(0, 2 * b, self.stride):
        #         for j in range(0, 2 * c, self.stride):
        #             cheated[k, i:i + self.stride, j:j + self.stride] = pooled_transpose_re[count]
        #             count = count + 1
        # return cheated
    
    def applying_sgd(self):
        pass

In [39]:
class softmax:
    def __init__(self):
        pass
    def expansion(self, t):
        (a,) = t.shape
        Y = np.zeros((a,10))
        for i in range(0,a):
            Y[i,t[i]] = 1
        return Y
    def forward_pass(self, z):
        self.z =  z
        (p,t) = self.z.shape
        self.a = np.zeros((p,t))
        for i in range(0,p):
            z_shift = self.z[i] - np.max(self.z[i])
            exp_sum = np.exp(z_shift).sum()
            for ii in range(0,t):
                self.a[i,ii] = np.exp(z_shift[ii]) / exp_sum
        return self.a
    def backprop(self, Y):
        y = self.expansion(Y)
        self.grad = (self.a - y)
        return self.grad
    def applying_sgd(self):
        pass

In [40]:
class relu:
    def __init__(self):
        pass

    def forward_pass(self, z):
        self.mask = (z > 0).astype(np.float32)
        return np.maximum(0, z)

    def backprop(self, grad_previous):
        return grad_previous * self.mask

    def applying_sgd(self):
        pass


class padding:
    def __init__(self, pad=1):
        self.pad = pad

    def forward_pass(self, data):
        return np.pad(data, ((0, 0), (0, 0), (self.pad, self.pad), (self.pad, self.pad)), mode='constant', constant_values=0)

    def backprop(self, y):
        return y[:, :, self.pad:y.shape[2] - self.pad, self.pad:y.shape[3] - self.pad]

    def applying_sgd(self):
        pass


class reshaping:
    def __init__(self):
        pass

    def forward_pass(self, a):
        self.shape_a = a.shape
        return a.reshape(a.shape[0], -1)

    def backprop(self, q):
        return q.reshape(self.shape_a)

    def applying_sgd(self):
        pass


class cross_entropy:
    def __init__(self):
        self.epsilon = 1e-10

    def expansion(self, t):
        (a,) = t.shape
        Y = np.zeros((a, 10))
        for i in range(0, a):
            Y[i, t[i]] = 1
        return Y

    def loss(self, A, Y):
        exp_Y = self.expansion(Y)
        (u, i) = A.shape
        loss_matrix = np.zeros((u, i))
        for j in range(u):
            for jj in range(i):
                if exp_Y[j, jj] == 0:
                    loss_matrix[j, jj] = np.log(1 - A[j, jj] + self.epsilon)
                else:
                    loss_matrix[j, jj] = np.log(A[j, jj] + self.epsilon)
        return ((-(loss_matrix.sum())) / u)


class accuracy:
    def __init__(self):
        pass

    def value(self, out, Y):
        self.out = np.argmax(out, axis=1)
        p = self.out.shape[0]
        total = 0
        for i in range(p):
            if Y[i] == self.out[i]:
                total += 1
        return total / p


class ConvNet:
    def __init__(self, Network):
        self.Network = Network

    def forward_pass(self, X):
        n = X
        for i in self.Network:
            n = i.forward_pass(n)
        return n

    def backprop(self, Y):
        m = Y
        for i in reversed(self.Network):
            m = i.backprop(m)

    def applying_sgd(self):
        for i in self.Network:
            i.applying_sgd()

In [ ]:
# Бүх MNIST сургалтын өгөгдлийг ашиглах



(Xtr, Ytr), (Xte, Yte) = Mnist.load_data()



X_training = Xtr / 255.0



Y_training = Ytr







# Тест өгөгдлийг бэлдэх



X_test = Xte / 255.0







# 4D input: (N, C, H, W)



X_training = X_training[:, np.newaxis, :, :]



X_test = X_test[:, np.newaxis, :, :]







al = 0.01







complete_NN = ConvNet([



    padding(1),



    ConvLayer(filter_dim=3, num_kernels=16, stride=1, pad=0, alpha=al),



    Pooling(pool_dim=2, stride=2),



    relu(),







    padding(1),



    ConvLayer(filter_dim=3, num_kernels=32, stride=1, pad=0, alpha=al),



    Pooling(pool_dim=2, stride=2),



    relu(),







    ConvLayer(filter_dim=3, num_kernels=168, stride=1, pad=0, alpha=al),



    relu(),







    reshaping(),



    Linear_Layer(168 * 5 * 5, 64, alpha=al),



    relu(),



    Linear_Layer(64, 10, alpha=al),



    softmax()



])







CE = cross_entropy()



acc = accuracy()



epochs = 1



batches = 64



total_samples = X_training.shape[0]







print(f"Нийт сургалтын өгөгдөл: {total_samples}")



print(f"Batch size: {batches}")



print(f"Batch тоо нэг epoch-т: {total_samples // batches}")



print(f"Epochs: {epochs}\\n")







for epoch in range(epochs):



    print(f"\\n{'=' * 60}")



    print(f"EPOCH {epoch + 1}/{epochs}")



    print(f"{'=' * 60}")







    k = 0



    batch_num = 0







    for j in range(batches, total_samples + 1, batches):



        batch_num += 1







        out = complete_NN.forward_pass(X_training[k:j])







        loss = CE.loss(out, Y_training[k:j])



        accur = acc.value(out, Y_training[k:j]) * 100







        if batch_num % 10 == 0 or batch_num == 1:



            print(f"Batch {batch_num:3d}/{total_samples // batches} \t Loss: {loss:.4f} \t Accuracy: {accur:.2f}%")







        complete_NN.backprop(Y_training[k:j])



        complete_NN.applying_sgd()







        k = j







    out_epoch = complete_NN.forward_pass(X_training)



    epoch_loss = CE.loss(out_epoch, Y_training)



    epoch_acc = acc.value(out_epoch, Y_training) * 100



    print(f"\\nEpoch {epoch + 1} дууслаа - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")







print(f"\\n{'=' * 60}")



print("ЭЦСИЙН ҮР ДҮН")



print(f"{'=' * 60}")







out = complete_NN.forward_pass(X_training)



final_train_loss = CE.loss(out, Y_training)



final_train_acc = acc.value(out, Y_training) * 100



print(f"Сургалтын багц: Loss = {final_train_loss:.4f}, Accuracy = {final_train_acc:.2f}%")







out_test = complete_NN.forward_pass(X_test)



test_acc = acc.value(out_test, Yte) * 100



print(f"Тестийн багц:   Accuracy = {test_acc:.2f}%")



print(f"{'=' * 60}")

Нийт сургалтын өгөгдөл: 60000
Batch size: 64
Batch тоо нэг epoch-т: 937
Epochs: 1\n
\n============================================================
EPOCH 1/1
Batch   1/937 	 Loss: 21.0313 	 Accuracy: 14.06%
Batch  10/937 	 Loss: 3.0437 	 Accuracy: 34.38%
Batch  20/937 	 Loss: 2.6126 	 Accuracy: 43.75%
Batch  30/937 	 Loss: 1.7182 	 Accuracy: 62.50%
Batch  40/937 	 Loss: 1.1500 	 Accuracy: 71.88%
Batch  50/937 	 Loss: 1.1624 	 Accuracy: 73.44%
Batch  60/937 	 Loss: 1.1322 	 Accuracy: 79.69%
Batch  70/937 	 Loss: 0.8209 	 Accuracy: 82.81%
Batch  80/937 	 Loss: 0.7855 	 Accuracy: 85.94%
Batch  90/937 	 Loss: 0.8746 	 Accuracy: 82.81%
Batch 100/937 	 Loss: 0.7185 	 Accuracy: 90.62%
Batch 110/937 	 Loss: 1.2014 	 Accuracy: 76.56%
Batch 120/937 	 Loss: 0.7954 	 Accuracy: 84.38%
Batch 130/937 	 Loss: 0.6088 	 Accuracy: 87.50%
Batch 140/937 	 Loss: 0.7175 	 Accuracy: 87.50%
